# Modules

First of all, since `gensim` is not installed by default in Colab we need to install it

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 32.8 MB/s eta 0:00:00


now we can load the modules

In [ ]:
# Base ----------------------------------
import string
import re
from collections import Counter

# NLTK ----------------------------------
import nltk
from nltk.corpus import stopwords
from nltk import FreqDist
from nltk.corpus import gutenberg
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.text import TextCollection
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline
for pkg in ["gutenberg", "punkt", "punkt_tab", "stopwords"]:
    nltk.download(pkg)

# spaCy ---------------------------------
import spacy
!python -m spacy download en_core_web_sm

# Gensim --------------------------------
from gensim.corpora import Dictionary
from gensim.models import TfidfModel
from gensim.similarities import MatrixSimilarity
from gensim.models.phrases import Phrases, Phraser   # Just for the N-grams

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 99.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


We may need a few cleaning utilities which are not too problematic and may affect the results

In [ ]:
def normalize_quotes(text: str):
    replacements = {
        "’": "'",
        "‘": "'",
        "“": '"',
        "”": '"'
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text

def clean_gutenberg_text(text: str):
    text = normalize_quotes(text)
    text = text.replace("\r\n", "\n")
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

def normalize(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text

# Corpus

Let's download "Moby Dick" of Herman Melville, and split it in sentences

In [ ]:
raw_text = gutenberg.raw("melville-moby_dick.txt")

clean_text = clean_gutenberg_text(raw_text)
sentences = sent_tokenize(clean_text)

Let's see the first sentences of our corpus

In [ ]:
sentences[:5]

['[Moby Dick by Herman Melville 1851] ETYMOLOGY.',
 '(Supplied by a Late Consumptive Usher to a Grammar School) The pale Usher--threadbare in coat, heart, body, and brain; I see him now.',
 'He was ever dusting his old lexicons and grammars, with a queer handkerchief, mockingly embellished with all the gay flags of all the known nations of the world.',
 'He loved to dust his old grammars; it somehow mildly reminded him of his mortality.',
 '"While you take in hand to school others, and to teach them by what name a whale-fish is to be called in our tongue leaving out, through ignorance, the letter H, which almost alone maketh the signification of the word, you deliver that which is not true."']

now, for this example I'm going to restric the corpus to a limited number of sentences

In [ ]:
sentences = sentences[2:50]

# Preprocessing

Let's normalize and tokenize our corpus

In [ ]:
normalized = [normalize(s) for s in sentences]
tokens = [nltk.word_tokenize(s) for s in normalized]
print(tokens)

[['he', 'was', 'ever', 'dusting', 'his', 'old', 'lexicons', 'and', 'grammars', 'with', 'a', 'queer', 'handkerchief', 'mockingly', 'embellished', 'with', 'all', 'the', 'gay', 'flags', 'of', 'all', 'the', 'known', 'nations', 'of', 'the', 'world'], ['he', 'loved', 'to', 'dust', 'his', 'old', 'grammars', 'it', 'somehow', 'mildly', 'reminded', 'him', 'of', 'his', 'mortality'], ['while', 'you', 'take', 'in', 'hand', 'to', 'school', 'others', 'and', 'to', 'teach', 'them', 'by', 'what', 'name', 'a', 'whalefish', 'is', 'to', 'be', 'called', 'in', 'our', 'tongue', 'leaving', 'out', 'through', 'ignorance', 'the', 'letter', 'h', 'which', 'almost', 'alone', 'maketh', 'the', 'signification', 'of', 'the', 'word', 'you', 'deliver', 'that', 'which', 'is', 'not', 'true'], ['hackluyt', 'whale'], ['sw', 'and', 'dan'], ['hval'], ['this', 'animal', 'is', 'named', 'from', 'roundness', 'or', 'rolling', 'for', 'in', 'dan'], ['hvalt', 'is', 'arched', 'or', 'vaulted'], ['websters', 'dictionary', 'whale'], [], ['

now we remove the stop-words

In [ ]:
stop_words = set(stopwords.words("english"))

tokens_no_sw = [
    [t for t in sent if t not in stop_words]
    for sent in tokens
]

print(tokens_no_sw)

[['ever', 'dusting', 'old', 'lexicons', 'grammars', 'queer', 'handkerchief', 'mockingly', 'embellished', 'gay', 'flags', 'known', 'nations', 'world'], ['loved', 'dust', 'old', 'grammars', 'somehow', 'mildly', 'reminded', 'mortality'], ['take', 'hand', 'school', 'others', 'teach', 'name', 'whalefish', 'called', 'tongue', 'leaving', 'ignorance', 'letter', 'h', 'almost', 'alone', 'maketh', 'signification', 'word', 'deliver', 'true'], ['hackluyt', 'whale'], ['sw', 'dan'], ['hval'], ['animal', 'named', 'roundness', 'rolling', 'dan'], ['hvalt', 'arched', 'vaulted'], ['websters', 'dictionary', 'whale'], [], ['immediately', 'dut'], ['ger'], ['wallen', 'walwian', 'roll', 'wallow'], ['richardsons', 'dictionary', 'ketos', 'greek'], ['cetus', 'latin'], ['whoel', 'anglosaxon'], ['hvalt', 'danish'], ['wal', 'dutch'], ['hwal', 'swedish'], ['whale', 'icelandic'], ['whale', 'english'], ['baleine', 'french'], ['ballena', 'spanish'], ['pekeenueenuee', 'fegee'], ['pekeenueenuee', 'erromangoan'], ['extract

# Modelling

Our corpus is ready to use, let's find out different representations.

In [ ]:
tokens_no_numbers = [
    [t for t in sent if t.isalpha()]
    for sent in tokens_no_sw
]

We  can find the most common words

In [ ]:
fd = FreqDist(t for sent in tokens_no_numbers for t in sent)
fd.most_common()[:10]

[('whale', 5),
 ('leviathan', 5),
 ('ye', 5),
 ('extracts', 4),
 ('shall', 4),
 ('great', 4),
 ('ever', 3),
 ('world', 3),
 ('take', 3),
 ('would', 3)]

## One-Hot Encoding

To find this representation we are first going to sort the tokens alphabetically and then produce the encoding

In [ ]:
vocab = sorted({t for sent in tokens_no_numbers for t in sent})
print("Vocabulary:", vocab)

one_hot = []
for sent in tokens_no_numbers:
    vec = [1 if w in sent else 0 for w in vocab]
    one_hot.append(vec)

print(one_hot)

Vocabulary: ['affording', 'allusions', 'almost', 'aloft', 'alone', 'altogether', 'ancient', 'anglosaxon', 'animal', 'anyways', 'appearing', 'appears', 'arched', 'authentic', 'authors', 'baleine', 'ballena', 'beast', 'belongest', 'besides', 'birds', 'bluntly', 'boat', 'book', 'bottomless', 'burrower', 'called', 'case', 'cetology', 'cetus', 'chaos', 'clear', 'clearing', 'cometh', 'coming', 'commentator', 'convivial', 'could', 'court', 'created', 'crooked', 'dan', 'danish', 'day', 'deep', 'deliver', 'devil', 'dictionary', 'dragon', 'dust', 'dusting', 'dut', 'dutch', 'earth', 'embellished', 'empty', 'english', 'entertaining', 'erromangoan', 'even', 'ever', 'every', 'extracts', 'eye', 'eyes', 'fancied', 'far', 'fare', 'feel', 'fegee', 'find', 'fish', 'flags', 'foul', 'french', 'friends', 'full', 'gabriel', 'gay', 'generally', 'generations', 'genesis', 'ger', 'glancing', 'glasses', 'go', 'god', 'goes', 'gone', 'gospel', 'grammars', 'great', 'greek', 'grow', 'grubworm', 'gulf', 'gulp', 'h', '

## TD-matrix

Let's now find the Term-Document matrix using NLTK

In [ ]:
collection = TextCollection(tokens_no_numbers)
vocab = sorted(set(collection))

print("Vocabulary size:", len(vocab))

tf_matrix = []
for doc in tokens_no_numbers:
    row = [doc.count(term) for term in vocab]
    tf_matrix.append(row)

print(tf_matrix)

Vocabulary size: 269
[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 

but we may also use `Collections`

In [ ]:
tf_matrix = []
for sent in tokens_no_numbers:
    counts = Counter(sent)
    tf_matrix.append([counts[w] for w in vocab])

print(tf_matrix)

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

### Using spaCy

Let's reduce the `raw_text` to just 10k tokens

In [ ]:
corpus = sent_tokenize(raw_text[1:10000])

In [ ]:
nlp = spacy.load("en_core_web_sm")
docs = list(nlp.pipe(corpus))

In [ ]:
tokens_spacy = [
    [t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop]
    for doc in docs
]

print(tokens_spacy)

[['moby', 'dick', 'herman', 'melville', 'etymology'], ['supply', 'late', 'consumptive', 'usher', 'grammar', 'school', 'pale', 'usher', 'threadbare', 'coat', 'heart', 'body', 'brain'], ['dust', 'old', 'lexicon', 'grammar', 'queer', 'handkerchief', 'mockingly', 'embellish', 'gay', 'flag', 'know', 'nation', 'world'], ['love', 'dust', 'old', 'grammar', 'mildly', 'remind', 'mortality'], ['hand', 'school', 'teach', 'whale', 'fish', 'call', 'tongue', 'leave', 'ignorance', 'letter', 'h', 'maketh', 'signification', 'word', 'deliver', 'true'], ['whale'], ['sw', 'dan'], ['hval'], ['animal', 'name', 'roundness', 'roll', 'dan'], ['hvalt', 'arch', 'vault'], ['dictionary', 'whale'], [], ['immediately', 'dut'], ['ger'], ['wallen', 'walw', 'ian', 'roll', 'wallow'], ['dictionary', 'ketos', 'greek'], ['cetus', 'latin'], ['whoel', 'anglo', 'saxon'], ['hvalt', 'danish'], ['wal', 'dutch'], ['hwal', 'swedish'], ['whale', 'icelandic'], ['whale', 'english'], ['baleine', 'french'], ['ballena', 'spanish'], ['pek

In [ ]:
dictionary = Dictionary(tokens_spacy)
corpus_bow = [dictionary.doc2bow(doc) for doc in tokens_spacy]

print("BoW:", corpus_bow)
print("Token IDs:", dictionary.token2id)

BoW: [[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1)], [(5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 2)], [(9, 1), (17, 1), (18, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1)], [(9, 1), (17, 1), (26, 1), (29, 1), (30, 1), (31, 1), (32, 1)], [(13, 1), (33, 1), (34, 1), (35, 1), (36, 1), (37, 1), (38, 1), (39, 1), (40, 1), (41, 1), (42, 1), (43, 1), (44, 1), (45, 1), (46, 1), (47, 1)], [(46, 1)], [(48, 1), (49, 1)], [(50, 1)], [(48, 1), (51, 1), (52, 1), (53, 1), (54, 1)], [(55, 1), (56, 1), (57, 1)], [(46, 1), (58, 1)], [], [(59, 1), (60, 1)], [(61, 1)], [(53, 1), (62, 1), (63, 1), (64, 1), (65, 1)], [(58, 1), (66, 1), (67, 1)], [(68, 1), (69, 1)], [(70, 1), (71, 1), (72, 1)], [(56, 1), (73, 1)], [(74, 1), (75, 1)], [(76, 1), (77, 1)], [(46, 1), (78, 1)], [(46, 1), (79, 1)], [(80, 1), (81, 1)], [(82, 1), (83, 1)], [(84, 1), (85, 2), (86, 1)], [(85, 2), (86, 1), (87, 1)], [(14, 1), (88, 1), (89,

## TF-IDF matrix

In [ ]:
tfidf = TfidfModel(corpus_bow)
corpus_tfidf = [tfidf[doc] for doc in corpus_bow]

print("TF–IDF:", corpus_tfidf)

TF–IDF: [[(0, np.float64(0.4472135954999579)), (1, np.float64(0.4472135954999579)), (2, np.float64(0.4472135954999579)), (3, np.float64(0.4472135954999579)), (4, np.float64(0.4472135954999579))], [(5, np.float64(0.23916133544551324)), (6, np.float64(0.23916133544551324)), (7, np.float64(0.2785176943189966)), (8, np.float64(0.2785176943189966)), (9, np.float64(0.21613934133960117)), (10, np.float64(0.21613934133960117)), (11, np.float64(0.2785176943189966)), (12, np.float64(0.23916133544551324)), (13, np.float64(0.23916133544551324)), (14, np.float64(0.23916133544551324)), (15, np.float64(0.2785176943189966)), (16, np.float64(0.5570353886379932))], [(9, np.float64(0.2310465972948233)), (17, np.float64(0.2556564317106455)), (18, np.float64(0.29772722152278325)), (19, np.float64(0.29772722152278325)), (20, np.float64(0.29772722152278325)), (21, np.float64(0.29772722152278325)), (22, np.float64(0.29772722152278325)), (23, np.float64(0.29772722152278325)), (24, np.float64(0.2977272215227832

we may turn the matrix into a dense representation for inspection (I leave that for you) but we stop here.

## Context Inspection

We can find that there exists a context for each word (even if we do not use it)  

In [ ]:
text = nltk.Text(t for sent in tokens_no_numbers for t in sent)
text.concordance("whale")

Displaying 5 of 5 matches:
ification word deliver true hackluyt whale sw dan hval animal named roundness r
t arched vaulted websters dictionary whale immediately dut ger wallen walwian r
 hvalt danish wal dutch hwal swedish whale icelandic whale english baleine fren
l dutch hwal swedish whale icelandic whale english baleine french ballena spani
ery case least take higgledypiggledy whale statements however authentic extract


Let's look for the sentences within the text itself (the sentences which contain the word itself)

In [ ]:
sentences_with_whale = []

for s in sentences:
    tokens = [w.lower() for w in nltk.word_tokenize(s) if w.isalpha()]
    if "whale" in tokens:
        sentences_with_whale.append(s)

for i in sentences_with_whale: print(i)

--HACKLUYT "WHALE.
--WEBSTER'S DICTIONARY "WHALE.
WHALE, ICELANDIC.
WHALE, ENGLISH.
Therefore you must not, in every case at least, take the higgledy-piggledy whale statements, however authentic, in these extracts, for veritable gospel cetology.


which is consistent with the previous result and clarifies the contextual sentences that appeared before

## Full BoW representation

In [ ]:
nlp = spacy.load("en_core_web_sm")

def preprocess_spacy(text: str, remove_stopwords: bool = True, use_lemmas: bool = True):
    doc = nlp(text)
    tokens = []
    for t in doc:
        if not t.is_alpha:
            continue
        if remove_stopwords and t.is_stop:
            continue
        tok = t.lemma_.lower() if use_lemmas else t.text.lower()
        tokens.append(tok)
    return tokens

docs_tokens = [preprocess_spacy(s, remove_stopwords=True, use_lemmas=True) for s in raw_text]
print("spaCy tokens:", docs_tokens)

dictionary = Dictionary(docs_tokens)
corpus_bow = [dictionary.doc2bow(doc) for doc in docs_tokens]

print("\nDictionary token2id:", dictionary.token2id)
print("BoW (sparse):", corpus_bow)

# Dense BoW vectors for inspection (This is optional)
# dense_bow = []
# for doc in corpus_bow:
#     vec = [0] * len(dictionary)
#     for term_id, count in doc:
#         vec[term_id] = count
#     dense_bow.append(vec)
#
# print("BoW (dense):", dense_bow)

tfidf_model = TfidfModel(corpus_bow, dictionary=dictionary)
corpus_tfidf = [tfidf_model[doc] for doc in corpus_bow]

print("\nTF–IDF (sparse):", corpus_tfidf)

index = MatrixSimilarity(corpus_tfidf, num_features=len(dictionary))
cosine_sims = [list(index[doc]) for doc in corpus_tfidf]

print("\nCosine similarity matrix:")
for row in cosine_sims:
    print(row)

# N-gram Model

In [ ]:
try:
    raw_text
except NameError:
    raw_text = """
    Call me Ishmael. Some years ago, never mind how long precisely,
    having little or no money in my purse, and nothing particular to interest me on shore,
    I thought I would sail about a little and see the watery part of the world.
    """

sentences = sent_tokenize(raw_text)
tokenized_sentences = [
    [w.lower() for w in word_tokenize(s) if w.isalpha()]
    for s in sentences
]

tokenized_sentences = tokenized_sentences[:200]

print("Example tokenized sentence:")
print(tokenized_sentences[0])

n = 3

train_data, vocab = padded_everygram_pipeline(n, tokenized_sentences)
lm = MLE(n)
lm.fit(train_data, vocab)

def generate_from_seed(model, seed, num_words=15):
    context = [w.lower() for w in seed.split()]
    generated = context.copy()

    for _ in range(num_words):
        next_word = model.generate(1, text_seed=generated[-(n-1):])
        if next_word == "</s>":
            break
        generated.append(next_word)

    return " ".join(generated)

# Examples
print("\nGenerated text:")
print(generate_from_seed(lm, seed="call me", num_words=20))
print(generate_from_seed(lm, seed="the sea", num_words=20))


Example tokenized sentence:
['moby', 'dick', 'by', 'herman', 'melville', 'etymology']

Generated text:
call me if is not true
the sea being then covered with them
